In [18]:
import pandas as pd
from openpyxl import load_workbook

# ① CSV読み込み
df = pd.read_csv("TCB.csv", header=None, encoding="shift_jis")

# ② Excel読み込み（テンプレート）
wb = load_workbook("プリントアウト.xlsx")
template_ws = wb.active

# 条件マッピング
mapping_g1 = {
    "N": "NCB",
    "T": "NUCT",
    "U": "TCB",
    "M": "飛島南",
    "K": "飛島北"
}

mapping_n1 = {
    "MT": "MITSUI-SOKO CO.,LTD.",
    "KG": "KAMIGUMI CO.,LTD.",
    "IK": "ISEWAN TERMINAL SERVICE CO.,LTD.",
    "FC": "FUJITRANS CORPORATION",
    "AU": "ASAHIUNYU CO.,LTD.",
    "MB": "MITSUBISHI LOGISTICS CORPORATION",
    "TK": "TOKAI KYOWA CO.,LTD",
    "NT": "NIPPON EXPRESS CO., LTD.",
    "MK": "MEIKO TRANS CO.,LTD."
}

# ③ 行ごとに処理
for i in range(len(df)):

    # 空行判定（A列が空なら終了）
    if pd.isna(df.iloc[i, 0]):
        break

    # 1行目は既存シート、それ以降はコピー
    if i == 0:
        ws = template_ws
        ws.title = f"Sheet{i+1}"
    else:
        ws = wb.copy_worksheet(template_ws)
        ws.title = f"Sheet{i+1}"

    # --- 通常転写 ---
    mapping = {
        0: "B4",
        1: "G14",
        2: "G13",
        3: "P4",
        7: "C7",
        8: "G10",
        9: "O10",
        10: "I11",
        12: "G12",
        14: "H15",
        15: "L15",
        16: "Q15",
        20: "P17",
    }

    for col, cell in mapping.items():
        ws[cell] = df.iloc[i, col]

    # --- 特殊処理 ---

    # FREE TIME
    ws["O11"] = f"FREE TIEM {df.iloc[i,11]} 迄"

    # G1関連（P5はG列のまま）
    g1_value = str(df.iloc[i, 6]).strip()
    ws["P5"] = mapping_g1.get(g1_value, g1_value)

    # ★ここを変更（G列 → R列）
    r_value = str(df.iloc[i, 17]).strip()
    ws["G16"] = r_value

    # N1 → P12
    n1_value = str(df.iloc[i, 13]).strip()
    ws["P12"] = mapping_n1.get(n1_value, n1_value)

    # V/W/X → A19
    v = df.iloc[i,21]
    w = df.iloc[i,22]
    x = df.iloc[i,23]
    ws["A19"] = f"{v}/{w}/{x}"

# ④ 保存
wb.save("##Dispatch order.xlsx")